# 01 — ENSANUT 2018: limpieza y extracción

**Fuente:** Encuesta Nacional de Salud y Nutrición 2018 — INEC  
**Archivo:** `1_BDD_ENS2018_f1_personas.csv`  
**Objetivo:** Extraer prevalencia de obesidad en adultos rurales (19-59 años) por provincia, como proxy de riesgo de diabetes donde el Estado no llega.

---

## Nota metodológica — ausencia de datos de diabetes

La **ENSANUT 2018 NO incluyó pruebas bioquímicas** (glucosa en sangre, HbA1c), a diferencia de la ENSANUT 2012. Por tanto, no existe variable de diagnóstico de diabetes ni glucosa en ningún archivo descargable de esta encuesta.

El módulo STEPS 2018 del MSP sí midió glucosa en adultos, pero su microdato no está disponible para descarga — solo existe el informe PDF con el dato agregado: **7,1% de adultos con glucosa elevada en ayunas**.

**Este es en sí mismo un hallazgo periodístico:** el Estado dejó de medir lo que más necesitaba medir, justo mientras la diabetes rural subía. Este punto forma parte de la narrativa de la historia.

La **prevalencia de diabetes** en el análisis cuantitativo se sustituye por **mortalidad por diabetes (CIE-10 E10-E14)** de los registros de Defunciones Generales INEC 2019-2024 — fuente más robusta y disponible a nivel provincial.

Lo que SÍ se extrae de ENSANUT: **obesidad en adultos rurales por provincia** — proxy del riesgo acumulado de diabetes en las poblaciones que el Estado menos atiende.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_DIR = Path('../raw/ENSANUT')
PROCESSED_DIR = Path('../processed')
PROCESSED_DIR.mkdir(exist_ok=True)

# Mapa de códigos provinciales INEC → nombre
PROVINCIAS = {
    1: 'AZUAY', 2: 'BOLIVAR', 3: 'CAÑAR', 4: 'CARCHI', 5: 'CHIMBORAZO',
    6: 'COTOPAXI', 7: 'EL ORO', 8: 'ESMERALDAS', 9: 'GUAYAS', 10: 'IMBABURA',
    11: 'LOJA', 12: 'LOS RIOS', 13: 'MANABI', 14: 'MORONA SANTIAGO', 15: 'NAPO',
    16: 'PASTAZA', 17: 'PICHINCHA', 18: 'TUNGURAHUA', 19: 'ZAMORA CHINCHIPE',
    20: 'GALAPAGOS', 21: 'SUCUMBIOS', 22: 'ORELLANA',
    23: 'SANTO DOMINGO DE LOS TSACHILAS', 24: 'SANTA ELENA'
}

## 1. Carga

In [2]:
f1 = pd.read_csv(
    RAW_DIR / '1_BDD_ENS2018_f1_personas.csv',
    encoding='latin-1',
    low_memory=False
)
print(f'Shape: {f1.shape}')
print(f'Area: {f1["area"].value_counts().to_dict()}')

Shape: (168747, 264)
Area: {'urbano': 102072, 'rural': 66675}


## 2. Inspección de variables clave

In [3]:
# Variables de obesidad — solo aplican a adultos 19-59 años
print('dobes19_59 (obesidad):', f1['dobes19_59'].value_counts().to_dict())
print('dspobes19_59 (sobrepeso+obesidad):', f1['dspobes19_59'].value_counts().to_dict())
print(f'Nulos dobes19_59: {f1["dobes19_59"].isna().sum()}')
print()
print('Rango de edad con variable no nula:')
print(f1[f1['dobes19_59'].notna()]['edadanios'].describe())

dobes19_59 (obesidad): {0.0: 58094, 1.0: 16692}
dspobes19_59 (sobrepeso+obesidad): {1.0: 47354, 0.0: 27432}
Nulos dobes19_59: 93961

Rango de edad con variable no nula:
count   74,786.00
mean        35.59
std         11.15
min         19.00
25%         26.00
50%         34.00
75%         44.00
max         59.00
Name: edadanios, dtype: float64


In [4]:
# prov=90 es agregado nacional — verificar
print('Provincias únicas:', sorted(f1['prov'].unique()))
print(f'Registros prov=90: {(f1["prov"] == 90).sum()}')

Provincias únicas: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 90]
Registros prov=90: 841


## 3. Filtros

- Área rural
- Adultos 19-59 años (rango donde aplican las variables de obesidad)
- Excluir prov=90 (agregado nacional, no es provincia)

In [5]:
n_total = len(f1)

df = f1[
    (f1['area'] == 'rural') &
    (f1['dobes19_59'].notna()) &
    (f1['prov'] != 90)
].copy()

print(f'Total original: {n_total}')
print(f'Después de filtros: {len(df)}')
print(f'Eliminados: {n_total - len(df)} ({(n_total - len(df))/n_total*100:.1f}%)')
print(f'Provincias representadas: {df["prov"].nunique()}')

Total original: 168747
Después de filtros: 27446
Eliminados: 141301 (83.7%)
Provincias representadas: 24


## 4. Nulos post-filtro

In [6]:
cols_analisis = ['prov', 'area', 'edadanios', 'sexo', 'etnia', 'dobes19_59', 'dspobes19_59', 'fexp']
nulos = df[cols_analisis].isnull().sum()
print('Nulos en variables de análisis:')
print(nulos[nulos > 0] if nulos.sum() > 0 else 'Ninguno')

Nulos en variables de análisis:
Ninguno


## 5. Prevalencia de obesidad rural por provincia

**Indicadores:**
- `prev_obesidad` — % adultos rurales 19-59 con obesidad (`dobes19_59 == 1`)
- `prev_sobrepeso_obesidad` — % adultos rurales 19-59 con sobrepeso u obesidad (`dspobes19_59 == 1`)

Las variables son binarias (0/1), por lo que la media directa es la prevalencia.

In [7]:
# Media ponderada usando factor de expansión muestral (fexp)
# La media simple puede diferir hasta 2,4 pp y cambiar el ranking provincial
def prom_ponderado(grupo, var):
    return (grupo[var] * grupo['fexp']).sum() / grupo['fexp'].sum()

resultado = []
for prov_id, grupo in df.groupby('prov'):
    resultado.append({
        'prov': prov_id,
        'n_adultos_rurales': len(grupo),
        'prev_obesidad': prom_ponderado(grupo, 'dobes19_59'),
        'prev_sobrepeso_obesidad': prom_ponderado(grupo, 'dspobes19_59')
    })

df_prov = pd.DataFrame(resultado)
df_prov['prev_obesidad'] = (df_prov['prev_obesidad'] * 100).round(1)
df_prov['prev_sobrepeso_obesidad'] = (df_prov['prev_sobrepeso_obesidad'] * 100).round(1)

df_prov['provincia'] = df_prov['prov'].map(PROVINCIAS)
df_prov = df_prov[['prov', 'provincia', 'n_adultos_rurales', 'prev_obesidad', 'prev_sobrepeso_obesidad']]
df_prov = df_prov.sort_values('prev_obesidad', ascending=False)

df_prov

,prov,provincia,n_adultos_rurales,prev_obesidad,prev_sobrepeso_obesidad
12,13,MANABI,1376,27.20,66.40
8,9,GUAYAS,879,26.70,68.70
3,4,CARCHI,935,25.30,69.20
19,20,GALAPAGOS,1590,25.00,69.50
6,7,EL ORO,972,24.70,67.30
16,17,PICHINCHA,398,24.60,68.80
7,8,ESMERALDAS,999,24.40,62.80
20,21,SUCUMBIOS,1349,22.60,63.90
22,23,SANTO DOMINGO DE LOS TSACHILAS,793,22.50,61.60
18,19,ZAMORA CHINCHIPE,1259,22.10,67.50


In [8]:
out = PROCESSED_DIR / 'ensanut_obesidad_rural_provincia.csv'
df_prov.to_csv(out, index=False)
print(f'Guardado: {out} ({len(df_prov)} filas)')

Guardado: ../processed/ensanut_obesidad_rural_provincia.csv (24 filas)


## 7. Resumen ETL

| Concepto | Valor |
|---|---|
| Registros totales f1 | 168.747 |
| Filtro: área rural + adultos 19-59 + prov válida | 27.446 |
| Eliminados | 141.301 (83,7%) |
| Provincias cubiertas | 24 |
| Output generado | ensanut_obesidad_rural_provincia.csv |

**Decisiones ETL:**
- Se excluye `prov=90` (código de agregado nacional, no corresponde a provincia)
- Las variables `dobes19_59` y `dspobes19_59` son nulas para menores de 19 y mayores de 59 — el filtro por `notna()` es equivalente a filtrar el rango de edad
- **Se aplica factor de expansión muestral (`fexp`) como ponderador.** La ENSANUT usa diseño muestral complejo y el `fexp` corrige el peso diferencial de cada observación. La diferencia vs. media simple puede alcanzar hasta 2,4 pp y modifica el ranking de provincias altas (sin peso lidera Carchi; con peso pasan arriba Manabí y Guayas).

**Limitación:** La ENSANUT 2018 no tiene variable de diagnóstico de diabetes ni glucosa en sangre. La obesidad es un proxy de riesgo, no de enfermedad confirmada. Ver nota metodológica al inicio del notebook.